1. Install & Imports

In [1]:
!pip install -q transformers scikit-learn

import os
import zipfile
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import CLIPProcessor, CLIPModel

2. Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


3. Paths

In [3]:
BASE_PATH = '/content/drive/MyDrive/memotion_project'
ZIP_PATH = os.path.join(BASE_PATH, 'dataset.zip')
EXTRACT_PATH = os.path.join(BASE_PATH, 'memotion_raw')

os.makedirs(EXTRACT_PATH, exist_ok=True)

4. Extract Dataset

In [4]:
if not os.listdir(EXTRACT_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)

5. Load CSV

In [26]:
CSV_PATH = None

for root, dirs, files in os.walk(EXTRACT_PATH):
    for f in files:
        if f == 'labels.csv':
            CSV_PATH = os.path.join(root, f)

df = pd.read_csv(CSV_PATH)

print(df.columns)

# ✅ DEFINE TEXT COLUMN HERE
TEXT_COL = 'text_ocr'

Index(['Unnamed: 0', 'image_name', 'text_ocr', 'text_corrected', 'humour',
       'sarcasm', 'offensive', 'motivational', 'overall_sentiment'],
      dtype='object')


In [27]:
print(TEXT_COL)

text_ocr


6. Label Setup (MULTI-LABEL)

In [38]:
from sklearn.preprocessing import LabelEncoder

LABEL_COLUMNS = ['humour', 'sarcasm', 'offensive', 'motivational']

encoders = {}

for col in LABEL_COLUMNS:
    # Convert everything to string first
    df[col] = df[col].astype(str)

    # Handle missing values
    df[col] = df[col].fillna("unknown")

    # Encode
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

    encoders[col] = le

print("Label encoding done")
print(df[LABEL_COLUMNS].head())

Label encoding done
   humour  sarcasm  offensive  motivational
0       1        0          1             1
1       2        0          1             0
2       3        1          1             1
3       3        2          3             0
4       1        3          3             1


In [37]:
print(df[LABEL_COLUMNS].dtypes)

humour          int64
sarcasm         int64
offensive       int64
motivational    int64
dtype: object


In [ ]:
7. Train / Val Split

In [21]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

8. Load CLIP

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "openai/clip-vit-base-patch32"

processor = CLIPProcessor.from_pretrained(model_name)
clip_model = CLIPModel.from_pretrained(model_name)
clip_model.to(device)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

9. Dataset Class

In [42]:
def load_image(path):
    try:
        return Image.open(path).convert("RGB")
    except:
        return None

class MemeDataset(torch.utils.data.Dataset):
    def __init__(self, df, text_col):
        self.df = df
        self.text_col = text_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        IMG_DIR = os.path.join(os.path.dirname(CSV_PATH), 'images')
        img_path = os.path.join(IMG_DIR, row['image_name'])
        if not os.path.exists(img_path):
          print("Missing image:", img_path)
          return None
        text = row[self.text_col]
        labels = torch.tensor(row[LABEL_COLUMNS].values.astype(float), dtype=torch.float)

        image = load_image(img_path)
        if image is None:
            return None

        inputs = processor(text=[text], images=image, return_tensors="pt", padding=True)

        item = {k: v.squeeze(0) for k, v in inputs.items()}
        item['labels'] = labels
        return item

10. Collate Function

In [24]:
def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None

    return {
        key: torch.stack([b[key] for b in batch])
        for key in batch[0]
    }

11. Dataloaders

In [40]:
train_loader = torch.utils.data.DataLoader(
    MemeDataset(train_df, TEXT_COL),
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = torch.utils.data.DataLoader(
    MemeDataset(val_df, TEXT_COL),
    batch_size=16,
    collate_fn=collate_fn
)

Multimodal Model

In [31]:
class MultimodalModel(nn.Module):
    def __init__(self, clip_model, num_labels):
        super().__init__()
        self.clip = clip_model

        self.fc = nn.Sequential(
            nn.Linear(512 * 2, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_labels)
        )

    def forward(self, input_ids, attention_mask, pixel_values):
        image_features = self.clip.get_image_features(pixel_values=pixel_values)
        text_features = self.clip.get_text_features(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        fused = torch.cat((image_features, text_features), dim=1)
        logits = self.fc(fused)

        return logits

13. Initialize Model

In [32]:
model = MultimodalModel(clip_model, len(LABEL_COLUMNS)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

14. Training Loop

In [33]:
def train_epoch(loader):
    model.train()
    total_loss = 0

    for batch in tqdm(loader):
        if batch is None:
            continue

        batch = {k: v.to(device) for k, v in batch.items()}

        logits = model(
            batch['input_ids'],
            batch['attention_mask'],
            batch['pixel_values']
        )

        loss = criterion(logits, batch['labels'])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

15. Validation Loop

In [43]:
def evaluate(loader):
    model.eval()

    preds, targets = [], []

    with torch.no_grad():
        for batch in loader:
            if batch is None:
                continue

            batch = {k: v.to(device) for k, v in batch.items()}

            logits = model(
                batch['input_ids'],
                batch['attention_mask'],
                batch['pixel_values']
            )

            pred = torch.sigmoid(logits).cpu().numpy()
            preds.append(pred)
            targets.append(batch['labels'].cpu().numpy())

    if len(preds) == 0:
      print("No valid validation samples!")
      return 0, 0

    preds = torch.tensor(preds).view(-1, len(LABEL_COLUMNS))
    targets = torch.tensor(targets).view(-1, len(LABEL_COLUMNS))

    # safer prediction
    preds = torch.argmax(preds, dim=1)
    targets = torch.argmax(targets, dim=1)

    acc = accuracy_score(targets, preds)
    f1 = f1_score(targets, preds, average='micro')

    return acc, f1

16. Train Model

In [44]:
EPOCHS = 3

for epoch in range(EPOCHS):
    loss = train_epoch(train_loader)
    acc, f1 = evaluate(val_loader)

    print(f"Epoch {epoch+1}")
    print(f"Loss: {loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f}")

100%|██████████| 350/350 [00:04<00:00, 85.92it/s]


No valid validation samples!
Epoch 1
Loss: 0.0000 | Acc: 0.0000 | F1: 0.0000


100%|██████████| 350/350 [00:04<00:00, 81.40it/s]


No valid validation samples!
Epoch 2
Loss: 0.0000 | Acc: 0.0000 | F1: 0.0000


100%|██████████| 350/350 [00:04<00:00, 80.91it/s]


No valid validation samples!
Epoch 3
Loss: 0.0000 | Acc: 0.0000 | F1: 0.0000


17. Save Model

In [45]:
torch.save(model.state_dict(), BASE_PATH + "/multimodal_model.pth")